# Environment Initialization 

In [ ]:
# Cell 1: Environment Setup
!pip install -q ultralytics opencv-python reportlab matplotlib pandas numpy pillow roboflow pyyaml
import zipfile
import os
import cv2
import urllib.request
import yaml
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from IPython.display import display
import torch
from ultralytics import YOLO

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

# Data Set

In [ ]:
DATASET_DIR = Path("./crack_defect_dataset")
ZIP_PATH = Path("./crack_defect_dataset.zip")

DATASET_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/crack-seg.zip"

if not DATASET_DIR.exists():
    print("Downloading surface defect dataset zip archive...")
    urllib.request.urlretrieve(DATASET_URL, ZIP_PATH)

    print("Extracting dataset files...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)

    os.remove(ZIP_PATH)
    print("Dataset ready!")

# Dynamically locate any .yaml configuration file inside the dataset folder
yaml_files = list(DATASET_DIR.rglob("*.yaml"))
if not yaml_files:
    raise FileNotFoundError(f"No YAML configuration file found inside {DATASET_DIR.resolve()}")

DATASET_YAML = yaml_files[0]

# Fix: Added explicit UTF-8 encoding to prevent Windows cp1252 decode errors
with open(DATASET_YAML, "r", encoding="utf-8") as f:
    yaml_config = yaml.safe_load(f)

raw_classes = yaml_config.get("names", {0: "Crack_Defect"})
if isinstance(raw_classes, list):
    DEFECT_CLASSES = {i: name for i, name in enumerate(raw_classes)}
else:
    DEFECT_CLASSES = {int(k): v for k, v in raw_classes.items()}

VAL_IMG_DIR = DATASET_DIR / "images" / "val"

print(f"Dataset directory: {DATASET_DIR.resolve()}")
print(f"Loaded Configuration File: {DATASET_YAML.name}")
print(f"Detected Defect Classes: {DEFECT_CLASSES}")

# YOLO Segmentation Model Training & Evaluation

In [ ]:
# Cell 3: Initialize Pre-trained Weights & Train Model on Real Roboflow Dataset
model = YOLO("yolo11n-seg.pt")

train_results = model.train(
    data=str(DATASET_YAML),
    epochs=250,
    imgsz=1024,  
    batch=16,
    optimizer="SGD",  
    patience=12,        
    device=0 if torch.cuda.is_available() else "cpu",
    project="industrial_qc_runs",
    name="roboflow_yolo_seg",
    save=True,
    verbose=True,
)

# Model Validation on Real Test Images
metrics = model.val(imgsz=1024)

results_data = {
    "(Metric)": [
        "ا(mAP@50)",
        "(mAP@50-95)",
    ],
    "(Box Detection)": [
        f"{metrics.box.map50 * 100:.2f}% ({metrics.box.map50:.4f})",
        f"{metrics.box.map * 100:.2f}% ({metrics.box.map:.4f})",
    ],
    "(Mask Segmentation)": [
        f"{metrics.seg.map50 * 100:.2f}% ({metrics.seg.map50:.4f})",
        f"{metrics.seg.map * 100:.2f}% ({metrics.seg.map:.4f})",
    ],
}

df_results = pd.DataFrame(results_data)

print("\n============================ model final results ============================")
display(df_results)

# Save Model

In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

weight_files = sorted(Path(".").rglob("best.pt"), key=lambda p: p.stat().st_mtime)

if weight_files:
   
    best_weights_path = weight_files[-1]
    print(f"Found latest model weights at: {best_weights_path}")

  
    direct_weights_path = Path("latest_best_model.pt")
    shutil.copy(best_weights_path, direct_weights_path)


    target_dir = best_weights_path.parent.parent

    
    archive_path = shutil.make_archive("latest_yolo_results", "zip", target_dir)
    print("Files zipped successfully!\n")

    display(FileLink(str(direct_weights_path)))
    display(FileLink("latest_yolo_results.zip"))
else:
    print("No 'best.pt' file found. Make sure model training has finished.")